In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path(
    r"E:\Pep_Projectdetails\selected_dataset\fat_india_working.csv"
)
print("Dataset file exists:", DATA_PATH.exists())
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn count:", len(df.columns))

Dataset file exists: True
Dataset shape: (1519, 837)


,firm_id,s3a,s3b,s3c,s4,s4a,s7,a3a,a3b,a3c,...,d8d,d8e,d8f,b14a5_decoded,b14a4_decoded,b14a6_decoded,b5a_decoded,b5f_decoded,b5g_decoded,b5h_decoded
0,2000030,1,2,2,2,NaN,12,100,0,0,...,Consumer preferences do not value quality of p...,Lack of technical skills,Too costly or not convinced of the economic be...,No,No,No,Yes,No,Yes,No
1,2000031,1,2,2,2,NaN,15,100,0,0,...,Lack of information,Too costly or not convinced of the economic be...,Consumer preferences do not value quality of p...,No,No,No,Yes,No,Yes,No
2,2000110,1,2,2,2,NaN,17,100,0,0,...,Too costly or not convinced of the economic be...,Lack of technical skills,It is difficult or costly to comply with gover...,No,No,No,Yes,Yes,Yes,No
3,2000515,1,2,2,2,NaN,210,100,0,0,...,Lack of technical skills,Consumer preferences do not value quality of p...,Lack of information,Yes,Yes,Yes,Yes,No,Yes,No
4,2000768,1,2,2,2,NaN,44,100,0,0,...,Too costly or not convinced of the economic be...,Consumer preferences do not value quality of p...,Lack of knowledge on acquiring,Yes,No,Yes,Yes,No,Yes,No



Column count: 837


In [4]:
# Step 2: Create Digital Payment Adoption target

payment_columns = [
    "b14a4_decoded",
    "b14a5_decoded",
    "b14a6_decoded"
]

# Check the payment columns
print("Payment columns:")
display(df[payment_columns].head())

# Create target
df["Digital_Payment_Adoption"] = (
    df[payment_columns]
    .eq("Yes")
    .any(axis=1)
    .astype(int)
)

# Show target distribution
print("\nDigital Payment Adoption distribution:")
print(df["Digital_Payment_Adoption"].value_counts())

print("\nPercentage:")
print(
    df["Digital_Payment_Adoption"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Payment columns:


,b14a4_decoded,b14a5_decoded,b14a6_decoded
0,No,No,No
1,No,No,No
2,No,No,No
3,Yes,Yes,Yes
4,No,Yes,Yes



Digital Payment Adoption distribution:
Digital_Payment_Adoption
1    1357
0     162
Name: count, dtype: int64

Percentage:
Digital_Payment_Adoption
1    89.34
0    10.66
Name: proportion, dtype: float64


C:\Users\Krishna\AppData\Local\Temp\ipykernel_5768\2336138439.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Digital_Payment_Adoption"] = (


In [5]:
# Step 3: Select final features

selected_features = [
    "s7",
    "a6",
    "d3c",
    "a5a",
    "b5f_decoded",
    "b5g_decoded",
    "d3a",
    "d7a",
    "d7b",
    "b13b5"
]

target = "Digital_Payment_Adoption"

# Create final ML dataset
model_df = df[selected_features + [target]].copy()

print("Final ML dataset shape:", model_df.shape)

print("\nSelected features:")
print(selected_features)

print("\nFirst 5 rows:")
display(model_df.head())

Final ML dataset shape: (1519, 11)

Selected features:
['s7', 'a6', 'd3c', 'a5a', 'b5f_decoded', 'b5g_decoded', 'd3a', 'd7a', 'd7b', 'b13b5']

First 5 rows:


,s7,a6,d3c,a5a,b5f_decoded,b5g_decoded,d3a,d7a,d7b,b13b5,Digital_Payment_Adoption
0,12,2004,0,11,No,Yes,2,1,2,2,0
1,15,1953,0,6,No,Yes,2,1,2,2,0
2,17,1981,1,3,Yes,Yes,1,1,1,2,0
3,210,2005,-9,4,No,Yes,2,2,2,2,1
4,44,2008,0,4,No,Yes,2,2,2,2,1


In [7]:
# Step 4: Check missing values and -9 codes

print("Missing values:")
display(model_df.isnull().sum())

print("\n-9 values:")
minus_9_counts = (model_df == -9).sum()
display(minus_9_counts[minus_9_counts > 0])

print("\nData types:")
print(model_df.dtypes)

Missing values:


s7                           0
a6                           0
d3c                          0
a5a                          0
b5f_decoded                 17
b5g_decoded                  0
d3a                          0
d7a                          0
d7b                          0
b13b5                        0
Digital_Payment_Adoption     0
dtype: int64


-9 values:


a6         7
d3c      182
d3a       61
d7b       23
b13b5      1
dtype: int64


Data types:
s7                          int64
a6                          int64
d3c                         int64
a5a                         int64
b5f_decoded                   str
b5g_decoded                   str
d3a                         int64
d7a                         int64
d7b                         int64
b13b5                       int64
Digital_Payment_Adoption    int64
dtype: object


In [8]:
# Step 5: Clean invalid -9 values

# Copy the dataset before cleaning
clean_df = model_df.copy()

# a6 is a numerical year.
# -9 means missing/unknown, so replace it with NaN.
clean_df["a6"] = clean_df["a6"].replace(-9, np.nan)

# b13b5 is a categorical survey response.
# -9 means missing/unknown.
clean_df["b13b5"] = clean_df["b13b5"].replace(-9, np.nan)

# Check the result
print("Remaining -9 values:")
minus_9_remaining = (clean_df == -9).sum()
display(minus_9_remaining[minus_9_remaining > 0])

print("\nMissing values after cleaning:")
display(clean_df.isnull().sum())

Remaining -9 values:


d3c    182
d3a     61
d7b     23
dtype: int64


Missing values after cleaning:


s7                           0
a6                           7
d3c                          0
a5a                          0
b5f_decoded                 17
b5g_decoded                  0
d3a                          0
d7a                          0
d7b                          0
b13b5                        1
Digital_Payment_Adoption     0
dtype: int64

In [9]:
# Step 6: Train/Test Split

from sklearn.model_selection import train_test_split

# Separate features and target
X = clean_df[selected_features].copy()
y = clean_df[target].copy()

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training data: (1215, 10)
Testing data : (304, 10)

Training target distribution:
Digital_Payment_Adoption
1    1085
0     130
Name: count, dtype: int64

Testing target distribution:
Digital_Payment_Adoption
1    272
0     32
Name: count, dtype: int64


In [10]:
# Step 7: Create preprocessing pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical features
numerical_features = [
    "s7",
    "a6"
]

# Categorical features
categorical_features = [
    "d3c",
    "a5a",
    "b5f_decoded",
    "b5g_decoded",
    "d3a",
    "d7a",
    "d7b",
    "b13b5"
]

# Numerical preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# Combine both
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [11]:
# Step 8: Train Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Create Logistic Regression pipeline
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

# Train the model
logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [12]:
# Step 9: Evaluate Logistic Regression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predict test data
y_pred_lr = logistic_model.predict(X_test)

# Calculate metrics
accuracy_lr = accuracy_score(y_test, y_pred_lr)

precision_lr = precision_score(
    y_test,
    y_pred_lr,
    pos_label=0,
    zero_division=0
)

recall_lr = recall_score(
    y_test,
    y_pred_lr,
    pos_label=0,
    zero_division=0
)

f1_lr = f1_score(
    y_test,
    y_pred_lr,
    pos_label=0,
    zero_division=0
)

print("Logistic Regression Results")
print("-" * 35)

print(f"Accuracy  : {accuracy_lr:.4f}")
print(f"Precision : {precision_lr:.4f}")
print(f"Recall    : {recall_lr:.4f}")
print(f"F1-Score  : {f1_lr:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, zero_division=0))

Logistic Regression Results
-----------------------------------
Accuracy  : 0.7599
Precision : 0.2796
Recall    : 0.8125
F1-Score  : 0.4160

Confusion Matrix:
[[ 26   6]
 [ 67 205]]

Classification Report:
              precision    recall  f1-score   support

           0       0.28      0.81      0.42        32
           1       0.97      0.75      0.85       272

    accuracy                           0.76       304
   macro avg       0.63      0.78      0.63       304
weighted avg       0.90      0.76      0.80       304



In [13]:
# Step 10: Train Random Forest

from sklearn.ensemble import RandomForestClassifier

# Create Random Forest pipeline
random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
random_forest_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [14]:
# Step 11: Evaluate Random Forest

# Predict test data
y_pred_rf = random_forest_model.predict(X_test)

# Calculate metrics
accuracy_rf = accuracy_score(y_test, y_pred_rf)

precision_rf = precision_score(
    y_test,
    y_pred_rf,
    pos_label=0,
    zero_division=0
)

recall_rf = recall_score(
    y_test,
    y_pred_rf,
    pos_label=0,
    zero_division=0
)

f1_rf = f1_score(
    y_test,
    y_pred_rf,
    pos_label=0,
    zero_division=0
)

print("Random Forest Results")
print("-" * 35)

print(f"Accuracy  : {accuracy_rf:.4f}")
print(f"Precision : {precision_rf:.4f}")
print(f"Recall    : {recall_rf:.4f}")
print(f"F1-Score  : {f1_rf:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

Random Forest Results
-----------------------------------
Accuracy  : 0.8059
Precision : 0.3448
Recall    : 0.9375
F1-Score  : 0.5042

Confusion Matrix:
[[ 30   2]
 [ 57 215]]

Classification Report:
              precision    recall  f1-score   support

           0       0.34      0.94      0.50        32
           1       0.99      0.79      0.88       272

    accuracy                           0.81       304
   macro avg       0.67      0.86      0.69       304
weighted avg       0.92      0.81      0.84       304



In [15]:
# Step 12: Find a better Random Forest decision threshold

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Probability of class 1 (Adopter)
y_prob_rf = random_forest_model.predict_proba(X_test)[:, 1]

results = []

for threshold in np.arange(0.30, 0.61, 0.01):
    y_pred_threshold = (y_prob_rf >= threshold).astype(int)

    results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(y_test, y_pred_threshold),
        "Precision": precision_score(
            y_test, y_pred_threshold, pos_label=0, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred_threshold, pos_label=0, zero_division=0
        ),
        "F1": f1_score(
            y_test, y_pred_threshold, pos_label=0, zero_division=0
        )
    })

threshold_results = pd.DataFrame(results)

print("Top thresholds by Accuracy:")
display(
    threshold_results
    .sort_values("Accuracy", ascending=False)
    .head(10)
)

Top thresholds by Accuracy:


,Threshold,Accuracy,Precision,Recall,F1
0,0.30,0.891447,0.480000,0.37500,0.421053
2,0.32,0.891447,0.481481,0.40625,0.440678
3,0.33,0.891447,0.483871,0.46875,0.476190
1,0.31,0.888158,0.461538,0.37500,0.413793
4,0.34,0.888158,0.468750,0.46875,0.468750
5,0.35,0.881579,0.441176,0.46875,0.454545
6,0.36,0.881579,0.444444,0.50000,0.470588
7,0.37,0.881579,0.450000,0.56250,0.500000
8,0.38,0.868421,0.409091,0.56250,0.473684
9,0.39,0.865132,0.404255,0.59375,0.481013


In [17]:
# Step 13: Evaluate Random Forest at the selected threshold

final_threshold = 0.33

y_pred_rf_final = (y_prob_rf >= final_threshold).astype(int)

print("Random Forest - Final Threshold:", final_threshold)
print("-" * 45)

print(f"Accuracy : {accuracy_score(y_test, y_pred_rf_final):.4f}")
print(
    f"Precision (Non-Adopter): "
    f"{precision_score(y_test, y_pred_rf_final, pos_label=0, zero_division=0):.4f}"
)
print(
    f"Recall (Non-Adopter): "
    f"{recall_score(y_test, y_pred_rf_final, pos_label=0, zero_division=0):.4f}"
)
print(
    f"F1 (Non-Adopter): "
    f"{f1_score(y_test, y_pred_rf_final, pos_label=0, zero_division=0):.4f}"
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_final))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_rf_final,
    zero_division=0
))

Random Forest - Final Threshold: 0.33
---------------------------------------------
Accuracy : 0.8914
Precision (Non-Adopter): 0.4839
Recall (Non-Adopter): 0.4688
F1 (Non-Adopter): 0.4762

Confusion Matrix:
[[ 15  17]
 [ 16 256]]

Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.47      0.48        32
           1       0.94      0.94      0.94       272

    accuracy                           0.89       304
   macro avg       0.71      0.70      0.71       304
weighted avg       0.89      0.89      0.89       304



In [18]:
# Final presentation metrics - Random Forest

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

final_threshold = 0.33

# Final predictions
y_pred_final = (y_prob_rf >= final_threshold).astype(int)

# Metrics for Class 1 = Adopter
accuracy_final = accuracy_score(y_test, y_pred_final)
precision_final = precision_score(y_test, y_pred_final, pos_label=1)
recall_final = recall_score(y_test, y_pred_final, pos_label=1)
f1_final = f1_score(y_test, y_pred_final, pos_label=1)

print("FINAL RANDOM FOREST MODEL")
print("=" * 45)
print(f"Threshold : {final_threshold}")
print(f"Accuracy  : {accuracy_final:.4f}")
print(f"Precision : {precision_final:.4f}")
print(f"Recall    : {recall_final:.4f}")
print(f"F1-Score  : {f1_final:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

print("\nComplete Classification Report:")
print(classification_report(
    y_test,
    y_pred_final,
    target_names=["Non-Adopter", "Adopter"],
    zero_division=0
))

FINAL RANDOM FOREST MODEL
Threshold : 0.33
Accuracy  : 0.8914
Precision : 0.9377
Recall    : 0.9412
F1-Score  : 0.9394

Confusion Matrix:
[[ 15  17]
 [ 16 256]]

Complete Classification Report:
              precision    recall  f1-score   support

 Non-Adopter       0.48      0.47      0.48        32
     Adopter       0.94      0.94      0.94       272

    accuracy                           0.89       304
   macro avg       0.71      0.70      0.71       304
weighted avg       0.89      0.89      0.89       304



In [1]:
import sys
print(sys.executable)

d:\FinTech_AI_Project\venv\Scripts\python.exe


In [2]:
print("Random Forest model available:", "random_forest_model" in globals())

Random Forest model available: False


In [19]:
import joblib
import json
from pathlib import Path

MODEL_DIR = Path(r"D:\FinTech_AI_Project\models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "final_random_forest_model.joblib"
THRESHOLD_PATH = MODEL_DIR / "final_random_forest_threshold.json"

# Save complete preprocessing + Random Forest pipeline
joblib.dump(random_forest_model, MODEL_PATH)

# Save prediction threshold separately
with open(THRESHOLD_PATH, "w") as f:
    json.dump({"threshold": final_threshold}, f, indent=2)

print("Final model saved successfully!")
print("Model path    :", MODEL_PATH)
print("Threshold path:", THRESHOLD_PATH)
print("Threshold     :", final_threshold)

Final model saved successfully!
Model path    : D:\FinTech_AI_Project\models\final_random_forest_model.joblib
Threshold path: D:\FinTech_AI_Project\models\final_random_forest_threshold.json
Threshold     : 0.33


In [20]:
import joblib
from pathlib import Path

MODEL_PATH = Path(r"D:\FinTech_AI_Project\models\final_random_forest_model.joblib")
THRESHOLD_PATH = Path(r"D:\FinTech_AI_Project\models\final_random_forest_threshold.json")

loaded_model = joblib.load(MODEL_PATH)

print("Model loaded successfully:", loaded_model is not None)
print("Model file exists:", MODEL_PATH.exists())
print("Threshold file exists:", THRESHOLD_PATH.exists())

Model loaded successfully: True
Model file exists: True
Threshold file exists: True
